> **Archived research notebook.** This file is not part of the runtime package. It may contain outdated methodology and must be rerun only with trusted data in an isolated environment. Current package artifacts use `allow_pickle=False`.


In [ ]:
# VXN-RAMNet Upgrade Notebook
# Backtracking-Based Branch Graph Learning
# Five-cell code
#
# Learning video:
#   backtracking_learning_route.mp4
#   College/root -> Junction -> first branch -> backtrack -> Junction -> second branch
#
# Query videos:
#   query_route_1.mp4
#   query_route_2.mp4
#
# Main fix:
#   Query branch evidence is selected from the strongest branch window,
#   not from early common/root frames.

# ============================================================
# CELL 1: SETUP + FRAME EXTRACTION
# ============================================================

import sys
import subprocess
from pathlib import Path
import shutil
import json
import time

def install_if_missing(package_name, import_name=None):
    try:
        __import__(import_name or package_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

install_if_missing("opencv-python", "cv2")
install_if_missing("numpy", "numpy")
install_if_missing("pandas", "pandas")
install_if_missing("pillow", "PIL")
install_if_missing("matplotlib", "matplotlib")

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt


# ----------------------------
# CONFIG
# ----------------------------

ROOT_DIR = Path(".")
VIDEOS_DIR = ROOT_DIR / "videos"

LEARNING_VIDEO_NAME = "backtracking_learning_route.mp4"

# You can add/remove query files here.
QUERY_VIDEO_NAMES = [
    "query_route_1.mp4",
    "query_route_2.mp4",
]

OUTPUT_DIR = ROOT_DIR / "vxn_backtracking_graph_outputs"

FRAMES_DIR = OUTPUT_DIR / "frames"
LEARNING_FRAMES_DIR = FRAMES_DIR / "learning"
QUERY_FRAMES_ROOT = FRAMES_DIR / "queries"

LEARNING_MAX_SECONDS = 45
QUERY_MAX_SECONDS = 20

# Your learning video has more route content, so it needs more frames.
LEARNING_FRAME_COUNT = 270

# Query videos are shorter.
QUERY_FRAME_COUNT = 120

VIDEO_EXTENSIONS = [".mp4", ".mov", ".avi", ".mkv", ".webm"]


# ----------------------------
# HELPERS
# ----------------------------

def find_video(video_name):
    possible_paths = [
        VIDEOS_DIR / video_name,
        ROOT_DIR / video_name,
        Path("/mnt/data") / video_name,
    ]

    for p in possible_paths:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"Could not find video: {video_name}\n"
        f"Place it in current folder, videos/, or /mnt/data."
    )


def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 30.0

    duration = total_frames / fps if fps > 0 else 0

    cap.release()

    return {
        "fps": float(fps),
        "total_frames": int(total_frames),
        "duration_seconds": float(duration),
    }


def extract_evenly_spaced_frames(video_path, output_folder, frame_count, max_seconds):
    output_folder.mkdir(parents=True, exist_ok=True)

    info = get_video_info(video_path)

    fps = info["fps"]
    total_frames = info["total_frames"]
    duration = info["duration_seconds"]

    usable_seconds = min(duration, max_seconds)
    usable_frame_count = int(usable_seconds * fps)
    usable_frame_count = min(usable_frame_count, total_frames)

    if usable_frame_count <= 0:
        raise ValueError(f"No usable frames in video: {video_path}")

    frame_indices = np.linspace(
        0,
        usable_frame_count - 1,
        frame_count,
        dtype=int
    )

    cap = cv2.VideoCapture(str(video_path))
    saved_paths = []

    for out_idx, frame_idx in enumerate(frame_indices, start=1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        success, frame = cap.read()

        if not success:
            print(f"Warning: could not read frame {frame_idx}")
            continue

        output_path = output_folder / f"frame_{out_idx:03d}.jpg"
        cv2.imwrite(str(output_path), frame)
        saved_paths.append(output_path)

    cap.release()

    return {
        "video_path": str(video_path),
        "fps": float(fps),
        "original_duration_seconds": float(duration),
        "used_duration_seconds": float(usable_seconds),
        "original_total_frames": int(total_frames),
        "used_frame_count": int(usable_frame_count),
        "requested_frames": int(frame_count),
        "saved_frames": int(len(saved_paths)),
        "output_folder": str(output_folder),
    }


def preview_video_frames(folder, title, count=12):
    frames = sorted(folder.glob("*.jpg"))

    if not frames:
        print(f"No frames found in {folder}")
        return

    indices = np.linspace(0, len(frames) - 1, min(count, len(frames)), dtype=int)

    cols = 4
    rows = int(np.ceil(len(indices) / cols))

    plt.figure(figsize=(16, rows * 3.2))

    for i, idx in enumerate(indices, start=1):
        img = Image.open(frames[idx]).convert("RGB")
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"idx {idx}", fontsize=9)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# ----------------------------
# CLEAN OUTPUT
# ----------------------------

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

LEARNING_FRAMES_DIR.mkdir(parents=True, exist_ok=True)
QUERY_FRAMES_ROOT.mkdir(parents=True, exist_ok=True)


# ----------------------------
# FIND AND EXTRACT VIDEOS
# ----------------------------

learning_video_path = find_video(LEARNING_VIDEO_NAME)
query_video_paths = [find_video(name) for name in QUERY_VIDEO_NAMES]

print("Learning video:", learning_video_path)
print("Query videos:")
for p in query_video_paths:
    print("-", p)

report = {
    "system": "VXN-RAMNet",
    "mode": "backtracking_branch_graph_learning",
    "learning_video": LEARNING_VIDEO_NAME,
    "query_videos": QUERY_VIDEO_NAMES,
    "learning_max_seconds": LEARNING_MAX_SECONDS,
    "query_max_seconds": QUERY_MAX_SECONDS,
    "learning_frame_count": LEARNING_FRAME_COUNT,
    "query_frame_count": QUERY_FRAME_COUNT,
    "created_at_unix": time.time(),
    "videos": {}
}

report["videos"]["learning"] = extract_evenly_spaced_frames(
    learning_video_path,
    LEARNING_FRAMES_DIR,
    LEARNING_FRAME_COUNT,
    LEARNING_MAX_SECONDS
)

for query_path in query_video_paths:
    query_name = query_path.stem
    query_folder = QUERY_FRAMES_ROOT / query_name

    report["videos"][query_name] = extract_evenly_spaced_frames(
        query_path,
        query_folder,
        QUERY_FRAME_COUNT,
        QUERY_MAX_SECONDS
    )

report_path = OUTPUT_DIR / "frame_extraction_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\nFrame extraction complete.")
print("Learning frames:", report["videos"]["learning"]["saved_frames"])

for query_path in query_video_paths:
    query_name = query_path.stem
    print(f"{query_name} frames:", report["videos"][query_name]["saved_frames"])

print("Report saved:", report_path.resolve())

preview_video_frames(
    LEARNING_FRAMES_DIR,
    "Learning Video: root → junction → first branch → backtrack → junction → second branch",
    count=12
)

for query_path in query_video_paths:
    query_name = query_path.stem
    preview_video_frames(
        QUERY_FRAMES_ROOT / query_name,
        f"Query Preview: {query_name}",
        count=12
    )



In [ ]:

# ============================================================
# CELL 2: EMBEDDING GENERATION
# ============================================================

import sys
import subprocess
from pathlib import Path
import json
import time

def install_if_missing(package_name, import_name=None):
    try:
        __import__(import_name or package_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

install_if_missing("tensorflow", "tensorflow")

import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image as keras_image


ROOT_DIR = Path(".")
OUTPUT_DIR = ROOT_DIR / "vxn_backtracking_graph_outputs"

FRAMES_DIR = OUTPUT_DIR / "frames"
LEARNING_FRAMES_DIR = FRAMES_DIR / "learning"
QUERY_FRAMES_ROOT = FRAMES_DIR / "queries"

EMBEDDINGS_FILE = OUTPUT_DIR / "vxn_backtracking_embeddings.npz"

MODEL_INPUT_SIZE = (224, 224)
BATCH_SIZE = 16
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".webp", ".bmp"]

if not LEARNING_FRAMES_DIR.exists():
    raise FileNotFoundError("Learning frames missing. Run Cell 1 first.")


def collect_frames(folder):
    return sorted([
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])


learning_frames = collect_frames(LEARNING_FRAMES_DIR)

query_folders = sorted([p for p in QUERY_FRAMES_ROOT.iterdir() if p.is_dir()])
query_frames_map = {folder.name: collect_frames(folder) for folder in query_folders}

print("Learning frames:", len(learning_frames))
print("Query folders:")
for name, frames in query_frames_map.items():
    print(f"- {name}: {len(frames)} frames")


print("\nLoading EfficientNetB0 frozen encoder...")

encoder = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

encoder.trainable = False

dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
_ = encoder.predict(dummy, verbose=0)

print("Encoder loaded.")


def l2_normalize_matrix(x):
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def load_image_array(img_path, flip=False):
    img = Image.open(img_path).convert("RGB")
    img = ImageOps.exif_transpose(img)

    if flip:
        img = ImageOps.mirror(img)

    img = img.resize(MODEL_INPUT_SIZE)
    arr = keras_image.img_to_array(img)
    return arr


def encode_frames(frame_paths, label, flip=False):
    embeddings = []
    paths = []

    start_time = time.time()

    for start in range(0, len(frame_paths), BATCH_SIZE):
        batch_paths = frame_paths[start:start + BATCH_SIZE]
        batch = [load_image_array(p, flip=flip) for p in batch_paths]
        batch = np.asarray(batch, dtype=np.float32)
        batch = preprocess_input(batch)

        emb = encoder.predict(batch, verbose=0).astype(np.float32)
        emb = l2_normalize_matrix(emb)

        embeddings.append(emb)
        paths.extend([str(p.as_posix()) for p in batch_paths])

        print(f"{label}: encoded {min(start + BATCH_SIZE, len(frame_paths))}/{len(frame_paths)} frames")

    embeddings = np.vstack(embeddings).astype(np.float32)

    elapsed = time.time() - start_time
    print(f"{label}: complete in {elapsed:.2f} sec")
    print(f"{label}: avg per frame {(elapsed / max(1, len(frame_paths))) * 1000:.2f} ms")

    return embeddings, np.array(paths)


print("\nEncoding learning frames...")
learning_embeddings, learning_paths = encode_frames(
    learning_frames,
    "LEARNING_ORIGINAL",
    flip=False
)

learning_embeddings_flip, _ = encode_frames(
    learning_frames,
    "LEARNING_FLIPPED",
    flip=True
)

query_embedding_dict = {}
query_embedding_flip_dict = {}
query_path_dict = {}

for query_name, frames in query_frames_map.items():
    print(f"\nEncoding query: {query_name}")

    q_emb, q_paths = encode_frames(
        frames,
        f"{query_name}_ORIGINAL",
        flip=False
    )

    q_flip, _ = encode_frames(
        frames,
        f"{query_name}_FLIPPED",
        flip=True
    )

    query_embedding_dict[query_name] = q_emb
    query_embedding_flip_dict[query_name] = q_flip
    query_path_dict[query_name] = q_paths

save_dict = {
    "learning_embeddings": learning_embeddings,
    "learning_embeddings_flip": learning_embeddings_flip,
    "learning_frame_paths": learning_paths,
    "query_names": np.array(list(query_embedding_dict.keys()))
}

for query_name in query_embedding_dict:
    save_dict[f"query_embeddings__{query_name}"] = query_embedding_dict[query_name]
    save_dict[f"query_embeddings_flip__{query_name}"] = query_embedding_flip_dict[query_name]
    save_dict[f"query_frame_paths__{query_name}"] = query_path_dict[query_name]

np.savez_compressed(EMBEDDINGS_FILE, **save_dict)

print("\nEmbeddings saved:")
print(EMBEDDINGS_FILE.resolve())
print("Learning embedding shape:", learning_embeddings.shape)



In [ ]:

# ============================================================
# CELL 3: AUTOMATIC BACKTRACKING GRAPH LEARNING
# ============================================================

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


ROOT_DIR = Path(".")
OUTPUT_DIR = ROOT_DIR / "vxn_backtracking_graph_outputs"

EMBEDDINGS_FILE = OUTPUT_DIR / "vxn_backtracking_embeddings.npz"
GRAPH_MEMORY_FILE = OUTPUT_DIR / "vxn_backtracking_graph_memory.npz"
GRAPH_METADATA_FILE = OUTPUT_DIR / "vxn_backtracking_graph_metadata.json"

# Important:
# This mapping is by exploration order.
# In the learning video:
#   first branch = the branch visited before backtracking
#   second branch = the branch visited after returning to the junction
#
# If your physical names are different, only change these two labels.
FIRST_BRANCH_NAME = "LEFT_BRANCH"
SECOND_BRANCH_NAME = "RIGHT_BRANCH"

COMMON_LABEL = 0
JUNCTION_LABEL = 1
FIRST_BRANCH_LABEL = 2
BACKTRACK_LABEL = 3
SECOND_BRANCH_LABEL = 4

COMPONENT_NAMES = np.array([
    "COMMON_PATH",
    "JUNCTION_A",
    FIRST_BRANCH_NAME,
    "BACKTRACK_TO_JUNCTION",
    SECOND_BRANCH_NAME
])

# Auto-detection ranges
FIRST_JUNCTION_SEARCH = (0.12, 0.48)
RETURN_JUNCTION_SEARCH = (0.48, 0.86)

JUNCTION_WINDOW = 5
JUNCTION_MEMORY_RADIUS = 6
MIN_JUNCTION_GAP_RATIO = 0.18

GOOD_JUNCTION_SCORE = 0.70
ACCEPTABLE_JUNCTION_SCORE = 0.56

GOOD_BACKTRACK_SCORE = 0.56
ACCEPTABLE_BACKTRACK_SCORE = 0.42

if not EMBEDDINGS_FILE.exists():
    raise FileNotFoundError("Embeddings missing. Run Cell 2 first.")


data = np.load(EMBEDDINGS_FILE, allow_pickle=False)

E = data["learning_embeddings"].astype(np.float32)
EF = data["learning_embeddings_flip"].astype(np.float32)
learning_frame_paths = data["learning_frame_paths"].tolist()

N = len(E)

print("Loaded learning embeddings.")
print("Frames:", N)
print("Embedding dimension:", E.shape[1])


def max_similarity_matrix(A, AF, B, BF):
    sims = [
        A @ B.T,
        AF @ B.T,
        A @ BF.T,
        AF @ BF.T
    ]
    return np.maximum.reduce(sims)


def moving_average(x, window=7):
    x = np.asarray(x, dtype=np.float32)

    if len(x) < window:
        return x

    pad = window // 2
    padded = np.pad(x, (pad, pad), mode="edge")
    return np.convolve(padded, np.ones(window) / window, mode="valid")


def safe_range(start, end, total):
    start = max(0, int(start))
    end = min(total - 1, int(end))

    if end < start:
        end = start

    return list(range(start, end + 1))


def segment_centroid(emb):
    if len(emb) == 0:
        return np.zeros((E.shape[1],), dtype=np.float32)

    c = np.mean(emb, axis=0, keepdims=True)
    norm = np.linalg.norm(c, axis=1, keepdims=True)
    norm[norm == 0] = 1.0

    return (c / norm)[0].astype(np.float32)


def window_similarity_score(S, i, j, radius=5):
    offsets = list(range(-radius, radius + 1))

    same_order_scores = []
    reverse_order_scores = []

    for d in offsets:
        i1 = i + d
        j_same = j + d
        j_rev = j - d

        if 0 <= i1 < S.shape[0] and 0 <= j_same < S.shape[1]:
            same_order_scores.append(S[i1, j_same])

        if 0 <= i1 < S.shape[0] and 0 <= j_rev < S.shape[1]:
            reverse_order_scores.append(S[i1, j_rev])

    same_score = float(np.mean(same_order_scores)) if same_order_scores else -1
    reverse_score = float(np.mean(reverse_order_scores)) if reverse_order_scores else -1

    return max(same_score, reverse_score), same_score, reverse_score


print("\nBuilding flip-aware self-similarity matrix...")
S = max_similarity_matrix(E, EF, E, EF)

# remove trivial diagonal area
ignore_radius = max(8, int(0.035 * N))
for i in range(N):
    lo = max(0, i - ignore_radius)
    hi = min(N, i + ignore_radius + 1)
    S[i, lo:hi] = -1.0

print("Self-similarity matrix:", S.shape)


def detect_junction_pair(S):
    first_start = int(FIRST_JUNCTION_SEARCH[0] * N)
    first_end = int(FIRST_JUNCTION_SEARCH[1] * N)

    return_start = int(RETURN_JUNCTION_SEARCH[0] * N)
    return_end = int(RETURN_JUNCTION_SEARCH[1] * N)

    min_gap = int(MIN_JUNCTION_GAP_RATIO * N)

    candidates = []

    for i in range(first_start, first_end):
        j_start = max(return_start, i + min_gap)

        for j in range(j_start, return_end):
            score, same_score, reverse_score = window_similarity_score(
                S,
                i,
                j,
                radius=JUNCTION_WINDOW
            )

            # prefer plausible locations:
            # first junction around 25-45%, return junction around 55-75%
            i_ratio = i / N
            j_ratio = j / N

            plausibility = (
                1.0
                - abs(i_ratio - 0.35) * 0.20
                - abs(j_ratio - 0.68) * 0.20
            )

            final_score = score * plausibility

            candidates.append({
                "first_junction_index": int(i),
                "return_junction_index": int(j),
                "score": float(score),
                "same_order_score": float(same_score),
                "reverse_order_score": float(reverse_score),
                "final_score": float(final_score),
            })

    candidates = sorted(candidates, key=lambda x: x["final_score"], reverse=True)

    if not candidates:
        raise ValueError("No junction candidates found.")

    return candidates[0], candidates[:10]


best_junction, top_junction_candidates = detect_junction_pair(S)

first_junction_index = int(best_junction["first_junction_index"])
return_junction_index = int(best_junction["return_junction_index"])
junction_score = float(best_junction["score"])

if junction_score >= GOOD_JUNCTION_SCORE:
    junction_confidence = "HIGH"
elif junction_score >= ACCEPTABLE_JUNCTION_SCORE:
    junction_confidence = "MEDIUM_REVIEW"
else:
    junction_confidence = "LOW_REVIEW"


def sequence_reverse_score(start, mid, end, sample_count=28):
    if mid <= start + 4 or end <= mid + 4:
        return -1.0

    a_idx = np.linspace(start, mid, sample_count, dtype=int)
    b_idx = np.linspace(mid, end, sample_count, dtype=int)[::-1]

    A = E[a_idx]
    AF = EF[a_idx]
    B = E[b_idx]
    BF = EF[b_idx]

    sim = max_similarity_matrix(A, AF, B, BF)
    diag = np.diag(sim)

    return float(np.mean(diag))


def detect_turnaround(first_junction_index, return_junction_index):
    gap = return_junction_index - first_junction_index

    margin = max(8, int(0.08 * N))

    search_start = first_junction_index + margin
    search_end = return_junction_index - margin

    if search_end <= search_start:
        fallback = (first_junction_index + return_junction_index) // 2
        return {
            "left_endpoint_index": int(fallback),
            "score": -1.0,
            "confidence": "FALLBACK_MIDDLE",
            "method": "fallback_middle"
        }

    candidates = []

    for mid in range(search_start, search_end):
        score = sequence_reverse_score(
            first_junction_index,
            mid,
            return_junction_index,
            sample_count=28
        )

        ratio = (mid - first_junction_index) / max(1, gap)

        # turn-around usually somewhere near middle of first junction -> return junction span
        balance = 1.0 - abs(ratio - 0.50) * 0.18

        candidates.append({
            "left_endpoint_index": int(mid),
            "score": float(score),
            "final_score": float(score * balance),
            "ratio": float(ratio)
        })

    candidates = sorted(candidates, key=lambda x: x["final_score"], reverse=True)
    best = candidates[0]

    if best["score"] >= GOOD_BACKTRACK_SCORE:
        confidence = "HIGH"
    elif best["score"] >= ACCEPTABLE_BACKTRACK_SCORE:
        confidence = "MEDIUM_REVIEW"
    else:
        confidence = "LOW_REVIEW"

    best["confidence"] = confidence
    best["method"] = "reverse_sequence_similarity"

    return best


turnaround_info = detect_turnaround(first_junction_index, return_junction_index)

turnaround_index = int(turnaround_info["left_endpoint_index"])
backtrack_score = float(turnaround_info["score"])
backtrack_confidence = turnaround_info["confidence"]


print("\n==============================")
print("BACKTRACKING GRAPH DETECTION")
print("==============================")
print("First junction index: ", first_junction_index)
print("Return junction index:", return_junction_index)
print("Junction score:       ", round(junction_score, 4))
print("Junction confidence:  ", junction_confidence)
print("Turnaround index:     ", turnaround_index)
print("Backtrack score:      ", round(backtrack_score, 4))
print("Backtrack confidence: ", backtrack_confidence)


# Build segments
common_start = 0
common_end = first_junction_index

first_branch_start = min(N - 1, first_junction_index + 1)
first_branch_end = max(first_branch_start, turnaround_index)

backtrack_start = min(N - 1, turnaround_index + 1)
backtrack_end = max(backtrack_start, return_junction_index)

second_branch_start = min(N - 1, return_junction_index + 1)
second_branch_end = N - 1

junction_first_start = max(0, first_junction_index - JUNCTION_MEMORY_RADIUS)
junction_first_end = min(N - 1, first_junction_index + JUNCTION_MEMORY_RADIUS)

junction_return_start = max(0, return_junction_index - JUNCTION_MEMORY_RADIUS)
junction_return_end = min(N - 1, return_junction_index + JUNCTION_MEMORY_RADIUS)

segments = {
    "COMMON_PATH": [int(common_start), int(common_end)],
    "JUNCTION_A_FIRST_VISIT": [int(junction_first_start), int(junction_first_end)],
    FIRST_BRANCH_NAME: [int(first_branch_start), int(first_branch_end)],
    "BACKTRACK_TO_JUNCTION": [int(backtrack_start), int(backtrack_end)],
    "JUNCTION_A_RETURN_VISIT": [int(junction_return_start), int(junction_return_end)],
    SECOND_BRANCH_NAME: [int(second_branch_start), int(second_branch_end)]
}

print("\nSegment split:")
for name, span in segments.items():
    print(f"{name}: {span[0]} -> {span[1]} ({span[1] - span[0] + 1} frames)")


common_idx = safe_range(common_start, common_end, N)

junction_idx = sorted(list(set(
    safe_range(junction_first_start, junction_first_end, N)
    + safe_range(junction_return_start, junction_return_end, N)
)))

first_branch_idx = safe_range(first_branch_start, first_branch_end, N)
backtrack_idx = safe_range(backtrack_start, backtrack_end, N)
second_branch_idx = safe_range(second_branch_start, second_branch_end, N)

memory_embeddings = np.vstack([
    E[common_idx],
    E[junction_idx],
    E[first_branch_idx],
    E[backtrack_idx],
    E[second_branch_idx],
]).astype(np.float32)

memory_embeddings_flip = np.vstack([
    EF[common_idx],
    EF[junction_idx],
    EF[first_branch_idx],
    EF[backtrack_idx],
    EF[second_branch_idx],
]).astype(np.float32)

memory_labels = np.concatenate([
    np.full(len(common_idx), COMMON_LABEL, dtype=np.int32),
    np.full(len(junction_idx), JUNCTION_LABEL, dtype=np.int32),
    np.full(len(first_branch_idx), FIRST_BRANCH_LABEL, dtype=np.int32),
    np.full(len(backtrack_idx), BACKTRACK_LABEL, dtype=np.int32),
    np.full(len(second_branch_idx), SECOND_BRANCH_LABEL, dtype=np.int32),
])

centroids = np.vstack([
    segment_centroid(E[common_idx]),
    segment_centroid(E[junction_idx]),
    segment_centroid(E[first_branch_idx]),
    segment_centroid(E[backtrack_idx]),
    segment_centroid(E[second_branch_idx]),
]).astype(np.float32)

np.savez_compressed(
    GRAPH_MEMORY_FILE,
    memory_embeddings=memory_embeddings,
    memory_embeddings_flip=memory_embeddings_flip,
    memory_labels=memory_labels,
    centroids=centroids,
    component_names=COMPONENT_NAMES,
    learning_embeddings=E,
    learning_embeddings_flip=EF,
    learning_frame_paths=np.array(learning_frame_paths),
    self_similarity_matrix=S.astype(np.float32),
    common_indices=np.array(common_idx, dtype=np.int32),
    junction_indices=np.array(junction_idx, dtype=np.int32),
    first_branch_indices=np.array(first_branch_idx, dtype=np.int32),
    backtrack_indices=np.array(backtrack_idx, dtype=np.int32),
    second_branch_indices=np.array(second_branch_idx, dtype=np.int32),
    first_junction_index=np.array([first_junction_index], dtype=np.int32),
    return_junction_index=np.array([return_junction_index], dtype=np.int32),
    turnaround_index=np.array([turnaround_index], dtype=np.int32),
)

metadata = {
    "system": "VXN-RAMNet",
    "mode": "backtracking_branch_graph_learning",
    "learning_path": "root -> junction -> first_branch -> backtrack -> junction -> second_branch",
    "model": "EfficientNetB0",
    "retraining_used": False,
    "flip_aware_similarity_used": True,
    "branch_label_note": "Branch names are assigned by exploration order. Change FIRST_BRANCH_NAME and SECOND_BRANCH_NAME if needed.",
    "graph": {
        "root": "ROOT",
        "common_path": "ROOT_TO_JUNCTION",
        "junction": "JUNCTION_A",
        "first_branch": FIRST_BRANCH_NAME,
        "second_branch": SECOND_BRANCH_NAME,
        "backtrack_segment": "BACKTRACK_TO_JUNCTION"
    },
    "detected_indices": {
        "first_junction_index": int(first_junction_index),
        "return_junction_index": int(return_junction_index),
        "turnaround_index": int(turnaround_index)
    },
    "confidence": {
        "junction_score": float(junction_score),
        "junction_confidence": junction_confidence,
        "backtrack_score": float(backtrack_score),
        "backtrack_confidence": backtrack_confidence
    },
    "segments": segments,
    "component_counts": {
        "COMMON_PATH": len(common_idx),
        "JUNCTION_A": len(junction_idx),
        FIRST_BRANCH_NAME: len(first_branch_idx),
        "BACKTRACK_TO_JUNCTION": len(backtrack_idx),
        SECOND_BRANCH_NAME: len(second_branch_idx)
    },
    "created_at_unix": time.time()
}

with open(GRAPH_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nGraph memory saved:", GRAPH_MEMORY_FILE.resolve())
print("Graph metadata saved:", GRAPH_METADATA_FILE.resolve())


# Visualize self similarity
plt.figure(figsize=(9, 8))
plt.imshow(S, aspect="auto")
plt.scatter([return_junction_index], [first_junction_index], c="red", marker="x", s=140, label="junction revisit")
plt.title("Learning Video Self-Similarity Matrix")
plt.xlabel("Later frame index")
plt.ylabel("Earlier frame index")
plt.colorbar(label="flip-aware cosine similarity")
plt.legend()
plt.tight_layout()
plt.show()


def show_segment(title, indices, max_images=8):
    if not indices:
        return

    if len(indices) <= max_images:
        show_idx = indices
    else:
        show_idx = np.linspace(indices[0], indices[-1], max_images, dtype=int).tolist()

    plt.figure(figsize=(len(show_idx) * 3, 4))

    for i, idx in enumerate(show_idx, start=1):
        img = Image.open(learning_frame_paths[idx]).convert("RGB")
        plt.subplot(1, len(show_idx), i)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"idx {idx}", fontsize=9)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_segment("COMMON PATH", common_idx)
show_segment(f"FIRST BRANCH MEMORY: {FIRST_BRANCH_NAME}", first_branch_idx)
show_segment("BACKTRACK MEMORY", backtrack_idx)
show_segment(f"SECOND BRANCH MEMORY: {SECOND_BRANCH_NAME}", second_branch_idx)
show_segment("JUNCTION MEMORY", junction_idx, max_images=10)



In [ ]:

# ============================================================
# CELL 4: MULTI-QUERY BRANCH CLASSIFICATION
# ============================================================

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


ROOT_DIR = Path(".")
OUTPUT_DIR = ROOT_DIR / "vxn_backtracking_graph_outputs"

EMBEDDINGS_FILE = OUTPUT_DIR / "vxn_backtracking_embeddings.npz"
GRAPH_MEMORY_FILE = OUTPUT_DIR / "vxn_backtracking_graph_memory.npz"
GRAPH_METADATA_FILE = OUTPUT_DIR / "vxn_backtracking_graph_metadata.json"

QUERY_REPORTS_DIR = OUTPUT_DIR / "query_reports"
QUERY_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not GRAPH_MEMORY_FILE.exists():
    raise FileNotFoundError("Graph memory missing. Run Cell 3 first.")

graph = np.load(GRAPH_MEMORY_FILE, allow_pickle=False)
emb_data = np.load(EMBEDDINGS_FILE, allow_pickle=False)

with open(GRAPH_METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

memory_embeddings = graph["memory_embeddings"].astype(np.float32)
memory_embeddings_flip = graph["memory_embeddings_flip"].astype(np.float32)
memory_labels = graph["memory_labels"].astype(np.int32)
centroids = graph["centroids"].astype(np.float32)
component_names = graph["component_names"].tolist()

query_names = emb_data["query_names"].tolist()

print("Loaded graph memory.")
print("Components:", component_names)
print("Queries:", query_names)


# Thresholds
MIN_BRANCH_SCORE = 0.58
MIN_BRANCH_GAP = 0.040

STRONG_BRANCH_SCORE = 0.72
STRONG_BRANCH_GAP = 0.070

UNKNOWN_SCORE = 0.54

COMMON_LABEL = 0
JUNCTION_LABEL = 1
FIRST_BRANCH_LABEL = 2
BACKTRACK_LABEL = 3
SECOND_BRANCH_LABEL = 4

FIRST_BRANCH_NAME = component_names[FIRST_BRANCH_LABEL]
SECOND_BRANCH_NAME = component_names[SECOND_BRANCH_LABEL]


def max_similarity_matrix(A, AF, B, BF):
    sims = [
        A @ B.T,
        AF @ B.T,
        A @ BF.T,
        AF @ BF.T
    ]
    return np.maximum.reduce(sims)


def component_score(q_emb, q_flip, mem_emb, mem_flip, centroid):
    if len(q_emb) == 0 or len(mem_emb) == 0:
        return 0.0, np.array([], dtype=np.float32)

    sim = max_similarity_matrix(q_emb, q_flip, mem_emb, mem_flip)

    best = np.max(sim, axis=1)

    top3 = []
    for row in sim:
        r = np.sort(row)[::-1]
        top3.append(np.mean(r[:min(3, len(r))]))

    top3 = np.array(top3, dtype=np.float32)

    c = centroid.reshape(1, -1)
    centroid_score = np.maximum(
        (q_emb @ c.T).reshape(-1),
        (q_flip @ c.T).reshape(-1)
    )

    per_frame = (
        0.50 * best
        + 0.30 * top3
        + 0.20 * centroid_score
    )

    return float(np.mean(per_frame)), per_frame


def branch_window_candidates(n):
    """
    This fixes your issue:
    The old code used frames near common-path end, so evidence stayed in root path.
    This function tests several later windows and chooses the one with strongest branch separation.
    """

    starts = sorted(list(set([
        int(0.45 * n),
        int(0.50 * n),
        int(0.55 * n),
        int(0.60 * n),
        int(0.65 * n),
        int(0.70 * n),
        int(0.75 * n),
    ])))

    windows = []

    for start in starts:
        for size_ratio in [0.20, 0.25, 0.30, 0.35]:
            size = max(12, int(size_ratio * n))
            end = min(n, start + size)

            if end - start >= 10:
                windows.append((start, end))

    # Always include tail windows.
    windows.append((int(0.60 * n), n))
    windows.append((int(0.65 * n), n))
    windows.append((int(0.70 * n), n))

    # remove duplicates
    windows = sorted(list(set(windows)))

    return windows


def classify_query(query_name):
    q_emb = emb_data[f"query_embeddings__{query_name}"].astype(np.float32)
    q_flip = emb_data[f"query_embeddings_flip__{query_name}"].astype(np.float32)
    q_paths = emb_data[f"query_frame_paths__{query_name}"].tolist()

    n = len(q_emb)

    first_mem = memory_embeddings[memory_labels == FIRST_BRANCH_LABEL]
    first_flip = memory_embeddings_flip[memory_labels == FIRST_BRANCH_LABEL]

    second_mem = memory_embeddings[memory_labels == SECOND_BRANCH_LABEL]
    second_flip = memory_embeddings_flip[memory_labels == SECOND_BRANCH_LABEL]

    common_mem = memory_embeddings[memory_labels == COMMON_LABEL]
    common_flip = memory_embeddings_flip[memory_labels == COMMON_LABEL]

    junction_mem = memory_embeddings[memory_labels == JUNCTION_LABEL]
    junction_flip = memory_embeddings_flip[memory_labels == JUNCTION_LABEL]

    first_centroid = centroids[FIRST_BRANCH_LABEL]
    second_centroid = centroids[SECOND_BRANCH_LABEL]
    common_centroid = centroids[COMMON_LABEL]
    junction_centroid = centroids[JUNCTION_LABEL]

    window_rows = []

    for start, end in branch_window_candidates(n):
        qe = q_emb[start:end]
        qf = q_flip[start:end]

        first_score, first_per = component_score(
            qe, qf,
            first_mem, first_flip,
            first_centroid
        )

        second_score, second_per = component_score(
            qe, qf,
            second_mem, second_flip,
            second_centroid
        )

        common_score, _ = component_score(
            qe, qf,
            common_mem, common_flip,
            common_centroid
        )

        junction_score, _ = component_score(
            qe, qf,
            junction_mem, junction_flip,
            junction_centroid
        )

        best_branch = max(first_score, second_score)
        branch_gap = abs(first_score - second_score)
        shared_score = max(common_score, junction_score)

        # choose window that is branch-like:
        # high branch score, clear gap, not only root/common.
        window_quality = (
            best_branch
            + 0.70 * branch_gap
            - 0.25 * shared_score
            + 0.03 * (start / max(1, n))
        )

        window_rows.append({
            "start": int(start),
            "end": int(end),
            "frame_count": int(end - start),
            "first_branch_score": float(first_score),
            "second_branch_score": float(second_score),
            "common_score": float(common_score),
            "junction_score": float(junction_score),
            "best_branch_score": float(best_branch),
            "branch_gap": float(branch_gap),
            "shared_score": float(shared_score),
            "window_quality": float(window_quality),
        })

    windows_df = pd.DataFrame(window_rows).sort_values(
        by="window_quality",
        ascending=False
    ).reset_index(drop=True)

    best_window = windows_df.iloc[0].to_dict()

    first_score = float(best_window["first_branch_score"])
    second_score = float(best_window["second_branch_score"])
    branch_gap = float(best_window["branch_gap"])
    best_branch_score = max(first_score, second_score)

    if best_branch_score < UNKNOWN_SCORE:
        prediction = "UNKNOWN_BRANCH"
        reason = "Branch evidence is too weak for both learned branches."
    elif best_branch_score >= STRONG_BRANCH_SCORE and branch_gap >= STRONG_BRANCH_GAP:
        if first_score > second_score:
            prediction = FIRST_BRANCH_NAME
            reason = f"Strong evidence for {FIRST_BRANCH_NAME}."
        else:
            prediction = SECOND_BRANCH_NAME
            reason = f"Strong evidence for {SECOND_BRANCH_NAME}."
    elif branch_gap < MIN_BRANCH_GAP:
        prediction = "UNCERTAIN_BRANCH"
        reason = "Branch scores are too close."
    elif best_branch_score < MIN_BRANCH_SCORE:
        prediction = "UNKNOWN_BRANCH"
        reason = "Best branch score is below minimum threshold."
    elif first_score > second_score:
        prediction = FIRST_BRANCH_NAME
        reason = f"{FIRST_BRANCH_NAME} score is higher."
    else:
        prediction = SECOND_BRANCH_NAME
        reason = f"{SECOND_BRANCH_NAME} score is higher."

    report = {
        "query_name": query_name,
        "prediction": prediction,
        "reason": reason,
        "best_window": {
            "start": int(best_window["start"]),
            "end": int(best_window["end"]),
            "frame_count": int(best_window["frame_count"])
        },
        "scores": {
            "first_branch_name": FIRST_BRANCH_NAME,
            "second_branch_name": SECOND_BRANCH_NAME,
            "first_branch_score": first_score,
            "second_branch_score": second_score,
            "branch_gap": branch_gap,
            "best_branch_score": float(best_branch_score),
            "common_score_in_selected_window": float(best_window["common_score"]),
            "junction_score_in_selected_window": float(best_window["junction_score"])
        },
        "thresholds": {
            "MIN_BRANCH_SCORE": MIN_BRANCH_SCORE,
            "MIN_BRANCH_GAP": MIN_BRANCH_GAP,
            "STRONG_BRANCH_SCORE": STRONG_BRANCH_SCORE,
            "STRONG_BRANCH_GAP": STRONG_BRANCH_GAP,
            "UNKNOWN_SCORE": UNKNOWN_SCORE
        },
        "all_windows": window_rows,
        "created_at_unix": time.time()
    }

    report_path = QUERY_REPORTS_DIR / f"{query_name}_classification_report.json"

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    return report, windows_df, q_paths


all_reports = []

for query_name in query_names:
    report, windows_df, q_paths = classify_query(query_name)
    all_reports.append(report)

    print("\n" + "=" * 60)
    print(f"QUERY RESULT: {query_name}")
    print("=" * 60)
    print("Prediction:", report["prediction"])
    print("Reason:", report["reason"])
    print("Selected window:", report["best_window"])
    print("First branch:", report["scores"]["first_branch_name"], round(report["scores"]["first_branch_score"], 4))
    print("Second branch:", report["scores"]["second_branch_name"], round(report["scores"]["second_branch_score"], 4))
    print("Branch gap:", round(report["scores"]["branch_gap"], 4))
    print("Common score in selected window:", round(report["scores"]["common_score_in_selected_window"], 4))
    print("Junction score in selected window:", round(report["scores"]["junction_score_in_selected_window"], 4))

    print("\nTop 5 evidence windows:")
    display(windows_df.head(5))

    # Show selected evidence frames.
    start = report["best_window"]["start"]
    end = report["best_window"]["end"]

    evidence_indices = np.linspace(start, end - 1, min(8, end - start), dtype=int).tolist()

    plt.figure(figsize=(len(evidence_indices) * 3, 4))

    for i, idx in enumerate(evidence_indices, start=1):
        img = Image.open(q_paths[idx]).convert("RGB")
        plt.subplot(1, len(evidence_indices), i)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"idx {idx}", fontsize=9)

    plt.suptitle(f"{query_name} - selected branch evidence - Prediction: {report['prediction']}")
    plt.tight_layout()
    plt.show()


summary_path = OUTPUT_DIR / "vxn_backtracking_all_query_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(all_reports, f, indent=2)

print("\nAll query summary saved:")
print(summary_path.resolve())



In [ ]:

# ============================================================
# CELL 5: FINAL VALIDATION SUMMARY
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT_DIR = Path(".")
OUTPUT_DIR = ROOT_DIR / "vxn_backtracking_graph_outputs"

GRAPH_METADATA_FILE = OUTPUT_DIR / "vxn_backtracking_graph_metadata.json"
SUMMARY_FILE = OUTPUT_DIR / "vxn_backtracking_all_query_summary.json"
FINAL_SUMMARY_FILE = OUTPUT_DIR / "vxn_backtracking_final_summary.json"

with open(GRAPH_METADATA_FILE, "r", encoding="utf-8") as f:
    graph_meta = json.load(f)

with open(SUMMARY_FILE, "r", encoding="utf-8") as f:
    query_reports = json.load(f)

rows = []

for r in query_reports:
    rows.append({
        "query": r["query_name"],
        "prediction": r["prediction"],
        "reason": r["reason"],
        "evidence_start": r["best_window"]["start"],
        "evidence_end": r["best_window"]["end"],
        "first_branch_name": r["scores"]["first_branch_name"],
        "first_branch_score": round(r["scores"]["first_branch_score"], 4),
        "second_branch_name": r["scores"]["second_branch_name"],
        "second_branch_score": round(r["scores"]["second_branch_score"], 4),
        "branch_gap": round(r["scores"]["branch_gap"], 4),
        "common_score_selected_window": round(r["scores"]["common_score_in_selected_window"], 4),
    })

df = pd.DataFrame(rows)

print("\n==============================")
print("FINAL BACKTRACKING GRAPH SUMMARY")
print("==============================")

print("\nDetected Graph:")
print("Root:", graph_meta["graph"]["root"])
print("Common path:", graph_meta["graph"]["common_path"])
print("Junction:", graph_meta["graph"]["junction"])
print("First branch:", graph_meta["graph"]["first_branch"])
print("Second branch:", graph_meta["graph"]["second_branch"])

print("\nDetected key indices:")
for k, v in graph_meta["detected_indices"].items():
    print(f"{k}: {v}")

print("\nConfidence:")
for k, v in graph_meta["confidence"].items():
    print(f"{k}: {v}")

print("\nSegments:")
for k, v in graph_meta["segments"].items():
    print(f"{k}: {v[0]} -> {v[1]}")

print("\nQuery Results:")
display(df)

final_summary = {
    "system": "VXN-RAMNet",
    "mode": "backtracking_branch_graph_learning",
    "graph": graph_meta["graph"],
    "detected_indices": graph_meta["detected_indices"],
    "confidence": graph_meta["confidence"],
    "segments": graph_meta["segments"],
    "query_results": rows,
    "important_note": (
        "This version fixes the previous issue where query evidence frames were selected too early "
        "from the common/root path. It now tests multiple late windows and selects the strongest "
        "branch evidence window."
    )
}

with open(FINAL_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print("\nFinal summary saved:")
print(FINAL_SUMMARY_FILE.resolve())

print("""
Implemented flow:

1. One learning video is used:
   root -> junction -> first branch -> backtrack -> junction -> second branch

2. The system extracts frames from the learning video.

3. EfficientNetB0 creates frozen visual embeddings.

4. A self-similarity matrix detects junction revisit and backtracking.

5. The video is split into:
   - common path
   - junction
   - first branch
   - backtracking segment
   - second branch

6. Each query video is classified using multiple candidate branch evidence windows.

7. The selected evidence frames are displayed from the actual branch area,
   not from the early root/common path.
""")
